Agentic AI Software Debugging and Test Automation System


In [ ]:

%pip install -q google-genai pytest


In [ ]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


In [ ]:

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Reply with exactly: Gemini connection test successful."
)

print(response.text)


Gemini connection test successful.


In [ ]:

buggy_code = """
def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)
"""

print("Buggy code:")
print(buggy_code)


Buggy code:

def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)



In [ ]:

prompt = f"""
You are a Python debugging assistant.

Analyze the following code:
{buggy_code}

Identify:
1. The bug.
2. Why it occurs.
3. A corrected version of the code.
4. One test case to verify the correction.

Do not execute the code. Return a clear explanation.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print(response.text)


Here is the analysis of the provided code:

### 1. The Bug
The function calculates the wrong statistical value for an average. Instead of dividing by the total count of numbers (`len(numbers)`), it divides by `len(numbers) - 1`. 

### 2. Why it Occurs
Dividing by `N - 1` (where $N$ is the sample size) is used when calculating the **sample variance** (Bessel's correction) to correct for bias. However, the standard mathematical **mean (average)** is calculated by dividing the sum of all elements by the total number of elements ($N$). Because of this incorrect denominator, the function will:
* Return an incorrect average for any list with 1 or more elements.
* Raise a `ZeroDivisionError` if the input is a list with a single element (`len([x]) - 1` results in `1 - 1 = 0`).

### 3. A Corrected Version of the Code
```python
def calculate_average(numbers):
    if not numbers:
        raise ValueError("Cannot calculate the average of an empty list.")
    total = sum(numbers)
    return total /

In [ ]:

def calculate_average(numbers):
    if not numbers:
        raise ValueError("Cannot calculate the average of an empty list.")

    total = sum(numbers)
    return total / len(numbers)


In [ ]:

%%writefile test_debugger.py

import pytest
from math import isclose

def calculate_average(numbers):
    if not numbers:
        raise ValueError("Cannot calculate the average of an empty list.")

    total = sum(numbers)
    return total / len(numbers)


def test_normal_average():
    assert isclose(calculate_average([2, 4, 6, 8]), 5.0)


def test_single_element():
    assert calculate_average([10]) == 10


def test_negative_numbers():
    assert calculate_average([-2, -4, -6]) == -4


def test_empty_list():
    with pytest.raises(ValueError):
        calculate_average([])


Writing test_debugger.py


In [ ]:

!python -m pytest test_debugger.py -v


============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.11
collected 4 items                                                              

test_debugger.py::test_normal_average PASSED                             [ 25%]
test_debugger.py::test_single_element PASSED                             [ 50%]
test_debugger.py::test_negative_numbers PASSED                           [ 75%]
test_debugger.py::test_empty_list PASSED                                 [100%]

============================== 4 passed in 0.02s ===============================


In [ ]:

%%writefile test_average.py

import pytest
from solution import calculate_average


def test_normal_average():
    assert calculate_average([2, 4, 6, 8]) == 5.0


def test_single_element():
    assert calculate_average([10]) == 10


def test_negative_numbers():
    assert calculate_average([-2, -4, -6]) == -4


def test_empty_list():
    with pytest.raises(ValueError):
        calculate_average([])


Writing test_average.py


In [ ]:

%%writefile solution.py

def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)


Writing solution.py


In [ ]:

!python -m pytest test_average.py -v


============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.11
collected 4 items                                                              

test_average.py::test_normal_average FAILED                              [ 25%]
test_average.py::test_single_element FAILED                              [ 50%]
test_average.py::test_negative_numbers FAILED                            [ 75%]
test_average.py::test_empty_list FAILED                                  [100%]

=================================== FAILURES ===================================
_____________________________ test_normal_average ______________________________

    def test_normal_average():
>       assert calculate_average([2, 4, 6, 8]) == 5.0
E       assert 6.666666666666667 == 5.0
E        +  where 6.6666666

In [14]:

import subprocess
from pathlib import Path
from google import genai
from google.colab import userdata

PROJECT_DIR = Path("/content")
SOLUTION_FILE = PROJECT_DIR / "solution.py"
TEST_FILE = PROJECT_DIR / "test_average.py"

# Read the original buggy code
buggy_code = SOLUTION_FILE.read_text()

# Run the tests and capture their failures
result = subprocess.run(
    ["python", "-m", "pytest", str(TEST_FILE), "-q"],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True,
    timeout=30
)

test_feedback = result.stdout + "\n" + result.stderr

print("Original test exit code:", result.returncode)
print("\nCaptured test feedback:\n")
print(test_feedback[:2500])


Original test exit code: 1

Captured test feedback:

FFFF                                                                     [100%]
=================================== FAILURES ===================================
_____________________________ test_normal_average ______________________________

    def test_normal_average():
>       assert calculate_average([2, 4, 6, 8]) == 5.0
E       assert 6.666666666666667 == 5.0
E        +  where 6.666666666666667 = calculate_average([2, 4, 6, 8])

test_average.py:7: AssertionError
_____________________________ test_single_element ______________________________

    def test_single_element():
>       assert calculate_average([10]) == 10
               ^^^^^^^^^^^^^^^^^^^^^^^

test_average.py:11: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

numbers = [10]

    def calculate_average(numbers):
        total = sum(numbers)
>       return total / (len(numbers) - 1)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
E       

In [15]:

prompt = f"""
You are a Python debugging agent.

Repair the code using the actual test failures.

ORIGINAL CODE:
{buggy_code}

TEST FAILURE OUTPUT:
{test_feedback}

Requirements:
- Keep the function name calculate_average.
- Calculate the correct arithmetic average.
- Raise ValueError for an empty list.
- Return the element for a single-element list.
- Do not change the tests.

Return only the complete Python code.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print(response.text)


```python
def calculate_average(numbers):
    if not numbers:
        raise ValueError("Cannot calculate the average of an empty list.")
    total = sum(numbers)
    return total / len(numbers)
```


In [16]:

from pathlib import Path

solution_file = Path("/content/solution.py")

fixed_code = response.text.strip()

# Remove Markdown code fences if Gemini includes them
if fixed_code.startswith("```"):
    lines = fixed_code.splitlines()
    if lines and lines[0].startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    fixed_code = "\n".join(lines)

# Basic validation before saving
compile(fixed_code, "solution.py", "exec")

if "def calculate_average(numbers):" not in fixed_code:
    raise ValueError("Expected function not found. Repair was not saved.")

solution_file.write_text(fixed_code)

print("Generated repair saved successfully.")
print("\nSaved code:\n")
print(fixed_code)


Generated repair saved successfully.

Saved code:

def calculate_average(numbers):
    if not numbers:
        raise ValueError("Cannot calculate the average of an empty list.")
    total = sum(numbers)
    return total / len(numbers)


In [17]:

import subprocess
import sys

result = subprocess.run(
    [
        sys.executable, "-m", "pytest",
        "/content/test_average.py", "-v"
    ],
    cwd="/content",
    capture_output=True,
    text=True,
    timeout=30
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode == 0:
    print("\nSUCCESS: All tests passed!")
else:
    print("\nREPAIR NOT VERIFIED: Some tests failed.")


============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.11
collecting ... collected 4 items

test_average.py::test_normal_average PASSED                              [ 25%]
test_average.py::test_single_element PASSED                              [ 50%]
test_average.py::test_negative_numbers PASSED                            [ 75%]
test_average.py::test_empty_list PASSED                                  [100%]

============================== 4 passed in 0.01s ===============================


SUCCESS: All tests passed!


In [18]:

from pathlib import Path
import subprocess
import sys
import re

# Project files
PROJECT_DIR = Path("/content")
SOLUTION_FILE = PROJECT_DIR / "solution.py"
TEST_FILE = PROJECT_DIR / "test_average.py"
BACKUP_FILE = PROJECT_DIR / "solution_original.py"

MAX_RETRIES = 3

# Preserve the initial buggy implementation once
if not BACKUP_FILE.exists():
    BACKUP_FILE.write_text(SOLUTION_FILE.read_text())

def run_tests():
    """Run the fixed test suite and return results."""
    result = subprocess.run(
        [
            sys.executable, "-m", "pytest",
            str(TEST_FILE), "-q"
        ],
        cwd=str(PROJECT_DIR),
        capture_output=True,
        text=True,
        timeout=30
    )
    return result.returncode, result.stdout + "\n" + result.stderr


def extract_python_code(text):
    """Extract Python from a Gemini response."""
    match = re.search(
        r"```(?:python)?\s*(.*?)```",
        text,
        flags=re.DOTALL | re.IGNORECASE
    )

    code = match.group(1).strip() if match else text.strip()
    compile(code, "candidate_solution.py", "exec")
    return code


def request_repair(code, feedback):
    """Ask Gemini for a candidate repair."""
    prompt = f"""
You are a Python debugging agent.

Repair the function using the supplied test results.

CURRENT CODE:
{code}

TEST RESULTS:
{feedback}

Requirements:
- Preserve the function name calculate_average.
- Return the arithmetic mean for a non-empty list.
- Raise ValueError for an empty list.
- Do not change or rewrite the tests.
- Return only the complete Python implementation.
"""
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    if not response.text:
        raise ValueError("Gemini returned an empty response.")

    return extract_python_code(response.text)


def run_debugging_agent():
    original_code = BACKUP_FILE.read_text()
    SOLUTION_FILE.write_text(original_code)

    for attempt in range(MAX_RETRIES + 1):
        code = SOLUTION_FILE.read_text()
        exit_code, feedback = run_tests()

        print(f"\n--- Test run {attempt + 1} ---")
        print(feedback[-1800:])

        if exit_code == 0:
            print("\nSUCCESS: All tests passed.")
            return True

        if attempt == MAX_RETRIES:
            break

        print(f"\nRepair attempt {attempt + 1} of {MAX_RETRIES}")

        try:
            candidate = request_repair(code, feedback)

            # Basic structural validation before saving
            if "def calculate_average(numbers):" not in candidate:
                raise ValueError(
                    "Candidate does not contain the expected function."
                )

            SOLUTION_FILE.write_text(candidate)
            print("Candidate saved. Running tests again...")

        except Exception as error:
            print("Repair attempt could not be applied:", error)

    print("\nSTOPPED: Retry limit reached; tests remain failing.")
    return False


print("Debugging agent is ready.")


Debugging agent is ready.


In [19]:

success = run_debugging_agent()
print("\nFinal status:", "PASS" if success else "FAIL")



--- Test run 1 ---
....                                                                     [100%]
4 passed in 0.02s



SUCCESS: All tests passed.

Final status: PASS


In [20]:

from pathlib import Path

solution_file = Path("/content/solution.py")
backup_file = Path("/content/solution_original.py")

# Restore the original code for a controlled experiment
solution_file.write_text(backup_file.read_text())

print("Original buggy solution restored.")
print("\nCurrent solution:\n")
print(solution_file.read_text())


Original buggy solution restored.

Current solution:

def calculate_average(numbers):
    if not numbers:
        raise ValueError("Cannot calculate the average of an empty list.")
    total = sum(numbers)
    return total / len(numbers)


In [21]:

from pathlib import Path

solution_file = Path("/content/solution.py")

buggy_code = '''def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)
'''

solution_file.write_text(buggy_code)

print("Buggy solution prepared:")
print(solution_file.read_text())


Buggy solution prepared:
def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)



In [22]:
success = run_debugging_agent()
print("\nFinal status:", "PASS" if success else "FAIL")


--- Test run 1 ---
....                                                                     [100%]
4 passed in 0.01s



SUCCESS: All tests passed.

Final status: PASS


In [23]:

from pathlib import Path

PROJECT_DIR = Path("/content")
SOLUTION_FILE = PROJECT_DIR / "solution.py"
BACKUP_FILE = PROJECT_DIR / "solution_original.py"

# Explicitly set the original buggy implementation as the baseline
buggy_code = '''def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)
'''

BACKUP_FILE.write_text(buggy_code)
SOLUTION_FILE.write_text(buggy_code)

print("Baseline reset.")
print("\nBackup file:")
print(BACKUP_FILE.read_text())

print("\nCurrent solution:")
print(SOLUTION_FILE.read_text())


Baseline reset.

Backup file:
def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)


Current solution:
def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)



In [24]:

import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "/content/test_average.py", "-q"],
    cwd="/content",
    capture_output=True,
    text=True,
    timeout=30
)

print(result.stdout)
print("Exit code:", result.returncode)


FFFF                                                                     [100%]
=================================== FAILURES ===================================
_____________________________ test_normal_average ______________________________

    def test_normal_average():
>       assert calculate_average([2, 4, 6, 8]) == 5.0
E       assert 6.666666666666667 == 5.0
E        +  where 6.666666666666667 = calculate_average([2, 4, 6, 8])

test_average.py:7: AssertionError
_____________________________ test_single_element ______________________________

    def test_single_element():
>       assert calculate_average([10]) == 10
               ^^^^^^^^^^^^^^^^^^^^^^^

test_average.py:11: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

numbers = [10]

    def calculate_average(numbers):
        total = sum(numbers)
>       return total / (len(numbers) - 1)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
E       ZeroDivisionError: division by zero

solution.py:3: Z

In [25]:
success = run_debugging_agent()
print("\nFinal status:", "PASS" if success else "FAIL")


--- Test run 1 ---
_____________ test_normal_average ______________________________

    def test_normal_average():
>       assert calculate_average([2, 4, 6, 8]) == 5.0
E       assert 6.666666666666667 == 5.0
E        +  where 6.666666666666667 = calculate_average([2, 4, 6, 8])

test_average.py:7: AssertionError
_____________________________ test_single_element ______________________________

    def test_single_element():
>       assert calculate_average([10]) == 10
               ^^^^^^^^^^^^^^^^^^^^^^^

test_average.py:11: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

numbers = [10]

    def calculate_average(numbers):
        total = sum(numbers)
>       return total / (len(numbers) - 1)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
E       ZeroDivisionError: division by zero

solution.py:3: ZeroDivisionError
____________________________ test_negative_numbers _____________________________

    def test_negative_numbers():
>       assert calculate_

In [26]:

import csv
from pathlib import Path

results_file = Path("/content/experiment_results.csv")

columns = [
    "experiment_id",
    "bug_type",
    "initial_tests_failed",
    "repair_attempts",
    "final_tests_passed",
    "final_status",
    "notes"
]

row = {
    "experiment_id": "EXP001",
    "bug_type": "Off-by-one error and missing empty-input validation",
    "initial_tests_failed": 4,
    "repair_attempts": 1,
    "final_tests_passed": 4,
    "final_status": "PASS",
    "notes": "Gemini-generated repair verified by pytest"
}

write_header = not results_file.exists()

with open(results_file, "a", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=columns)

    if write_header:
        writer.writeheader()

    writer.writerow(row)

print("Experiment result saved to:", results_file)
print(results_file.read_text())


Experiment result saved to: /content/experiment_results.csv
experiment_id,bug_type,initial_tests_failed,repair_attempts,final_tests_passed,final_status,notes
EXP001,Off-by-one error and missing empty-input validation,4,1,4,PASS,Gemini-generated repair verified by pytest



In [27]:

from pathlib import Path

Path("/content/solution.py").write_text('''def is_adult(age):
    return age > 18
''')

Path("/content/test_average.py").write_text('''import pytest
from solution import is_adult

def test_adult_age():
    assert is_adult(25) is True

def test_exactly_18():
    assert is_adult(18) is True

def test_under_18():
    assert is_adult(17) is False
''')

print("Second bug case prepared: incorrect conditional logic.")


Second bug case prepared: incorrect conditional logic.


In [28]:

import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "/content/test_average.py", "-q"],
    cwd="/content",
    capture_output=True,
    text=True,
    timeout=30
)

print(result.stdout)
print("Exit code:", result.returncode)


.F.                                                                      [100%]
=================================== FAILURES ===================================
_______________________________ test_exactly_18 ________________________________

    def test_exactly_18():
>       assert is_adult(18) is True
E       assert False is True
E        +  where False = is_adult(18)

test_average.py:8: AssertionError
=========================== short test summary info ============================
FAILED test_average.py::test_exactly_18 - assert False is True
1 failed, 2 passed in 0.03s

Exit code: 1


In [29]:

def request_repair(code, feedback):
    """Ask Gemini to repair a general Python function."""

    prompt = f"""
You are an autonomous Python debugging agent.

Repair the provided Python code using the test failure feedback.

CURRENT CODE:
{code}

TEST FAILURE OUTPUT:
{feedback}

Requirements:
- Identify the actual bug from the code and test feedback.
- Preserve the intended function names and behavior.
- Do not modify the test file.
- Return the complete corrected Python implementation only.
- Do not include Markdown code fences or explanations.
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    if not response.text:
        raise ValueError("Gemini returned an empty response.")

    return extract_python_code(response.text)


In [30]:

def run_debugging_agent():
    """Run tests and attempt up to MAX_RETRIES repairs."""

    # Start from the current solution, not an old backup.
    for attempt in range(MAX_RETRIES + 1):
        code = SOLUTION_FILE.read_text()
        exit_code, feedback = run_tests()

        print(f"\n--- Test run {attempt + 1} ---")
        print(feedback[-1500:])

        if exit_code == 0:
            print("\nSUCCESS: All tests passed.")
            return True

        if attempt == MAX_RETRIES:
            break

        print(f"\nRepair attempt {attempt + 1} of {MAX_RETRIES}")

        try:
            candidate = request_repair(code, feedback)

            # Validate syntax before saving
            compile(candidate, "candidate_solution.py", "exec")

            # Save candidate and let pytest determine whether it works
            SOLUTION_FILE.write_text(candidate)
            print("Candidate saved. Running tests again...")

        except Exception as error:
            print("Repair attempt failed:", error)

    print("\nSTOPPED: Retry limit reached or repair attempts failed.")
    return False


print("Generic debugging agent is ready.")


Generic debugging agent is ready.


In [31]:

success = run_debugging_agent()
print("\nFinal status:", "PASS" if success else "FAIL")



--- Test run 1 ---
.F.                                                                      [100%]
=================================== FAILURES ===================================
_______________________________ test_exactly_18 ________________________________

    def test_exactly_18():
>       assert is_adult(18) is True
E       assert False is True
E        +  where False = is_adult(18)

test_average.py:8: AssertionError
=========================== short test summary info ============================
FAILED test_average.py::test_exactly_18 - assert False is True
1 failed, 2 passed in 0.03s



Repair attempt 1 of 3
Candidate saved. Running tests again...

--- Test run 2 ---
...                                                                      [100%]
3 passed in 0.01s



SUCCESS: All tests passed.

Final status: PASS


In [32]:

import csv
from pathlib import Path

results_file = Path("/content/experiment_results.csv")

row = {
    "experiment_id": "EXP002",
    "bug_type": "Incorrect conditional logic",
    "initial_tests_failed": 1,
    "repair_attempts": 1,
    "final_tests_passed": 3,
    "final_status": "PASS",
    "notes": "Gemini changed the age boundary condition; pytest verified the repair"
}

with open(results_file, "a", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "experiment_id",
            "bug_type",
            "initial_tests_failed",
            "repair_attempts",
            "final_tests_passed",
            "final_status",
            "notes"
        ]
    )
    writer.writerow(row)

print("Second experiment recorded.")
print(results_file.read_text())


Second experiment recorded.
experiment_id,bug_type,initial_tests_failed,repair_attempts,final_tests_passed,final_status,notes
EXP001,Off-by-one error and missing empty-input validation,4,1,4,PASS,Gemini-generated repair verified by pytest
EXP002,Incorrect conditional logic,1,1,3,PASS,Gemini changed the age boundary condition; pytest verified the repair



In [33]:

import csv
from pathlib import Path

results_file = Path("/content/experiment_results.csv")

with open(results_file, newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))

total = len(rows)
successful = sum(row["final_status"] == "PASS" for row in rows)
total_repairs = sum(int(row["repair_attempts"]) for row in rows)
total_initial_failures = sum(
    int(row["initial_tests_failed"]) for row in rows
)
total_final_passes = sum(
    int(row["final_tests_passed"]) for row in rows
)

success_rate = (successful / total * 100) if total else 0

print("=== AGENT EVALUATION SUMMARY ===")
print("Experiments:", total)
print("Successful experiments:", successful)
print(f"Experiment success rate: {success_rate:.1f}%")
print("Total repair attempts:", total_repairs)
print("Initial test failures:", total_initial_failures)
print("Final passing test counts:", total_final_passes)

print("\n=== INDIVIDUAL RESULTS ===")
for row in rows:
    print(
        f'{row["experiment_id"]}: {row["bug_type"]} | '
        f'{row["final_status"]} | '
        f'{row["repair_attempts"]} repair attempt(s)'
    )


=== AGENT EVALUATION SUMMARY ===
Experiments: 2
Successful experiments: 2
Experiment success rate: 100.0%
Total repair attempts: 2
Initial test failures: 5
Final passing test counts: 7

=== INDIVIDUAL RESULTS ===
EXP001: Off-by-one error and missing empty-input validation | PASS | 1 repair attempt(s)
EXP002: Incorrect conditional logic | PASS | 1 repair attempt(s)


In [34]:

from pathlib import Path

Path("/content/solution.py").write_text('''def calculate_total(prices):
    total = sum(price for price in prices)
    return tota
''')

Path("/content/test_average.py").write_text('''from solution import calculate_total

def test_normal_total():
    assert calculate_total([10, 20, 30]) == 60

def test_single_price():
    assert calculate_total([15]) == 15

def test_empty_prices():
    assert calculate_total([]) == 0
''')

print("Third bug case prepared: undefined variable.")


Third bug case prepared: undefined variable.


In [35]:

import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "/content/test_average.py", "-q"],
    cwd="/content",
    capture_output=True,
    text=True,
    timeout=30
)

print(result.stdout)
print("Exit code:", result.returncode)


FFF                                                                      [100%]
=================================== FAILURES ===================================
______________________________ test_normal_total _______________________________

    def test_normal_total():
>       assert calculate_total([10, 20, 30]) == 60
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

test_average.py:4: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

prices = [10, 20, 30]

    def calculate_total(prices):
        total = sum(price for price in prices)
>       return tota
               ^^^^
E       NameError: name 'tota' is not defined

solution.py:3: NameError
______________________________ test_single_price _______________________________

    def test_single_price():
>       assert calculate_total([15]) == 15
               ^^^^^^^^^^^^^^^^^^^^^

test_average.py:7: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

prices = [15]

    d

In [36]:
success = run_debugging_agent()
print("\nFinal status:", "PASS" if success else "FAIL")


--- Test run 1 ---
:
        total = sum(price for price in prices)
>       return tota
               ^^^^
E       NameError: name 'tota' is not defined

solution.py:3: NameError
______________________________ test_single_price _______________________________

    def test_single_price():
>       assert calculate_total([15]) == 15
               ^^^^^^^^^^^^^^^^^^^^^

test_average.py:7: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

prices = [15]

    def calculate_total(prices):
        total = sum(price for price in prices)
>       return tota
               ^^^^
E       NameError: name 'tota' is not defined

solution.py:3: NameError
______________________________ test_empty_prices _______________________________

    def test_empty_prices():
>       assert calculate_total([]) == 0
               ^^^^^^^^^^^^^^^^^^^

test_average.py:10: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

prices = []

    def calcula

In [37]:

import csv
from pathlib import Path

results_file = Path("/content/experiment_results.csv")

row = {
    "experiment_id": "EXP003",
    "bug_type": "Undefined variable (NameError)",
    "initial_tests_failed": 3,
    "repair_attempts": 1,
    "final_tests_passed": 3,
    "final_status": "PASS",
    "notes": "Gemini-generated repair verified by pytest"
}

with open(results_file, "a", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "experiment_id",
            "bug_type",
            "initial_tests_failed",
            "repair_attempts",
            "final_tests_passed",
            "final_status",
            "notes"
        ]
    )
    writer.writerow(row)

print("Experiment 3 recorded successfully.")
print(results_file.read_text())


Experiment 3 recorded successfully.
experiment_id,bug_type,initial_tests_failed,repair_attempts,final_tests_passed,final_status,notes
EXP001,Off-by-one error and missing empty-input validation,4,1,4,PASS,Gemini-generated repair verified by pytest
EXP002,Incorrect conditional logic,1,1,3,PASS,Gemini changed the age boundary condition; pytest verified the repair
EXP003,Undefined variable (NameError),3,1,3,PASS,Gemini-generated repair verified by pytest



In [38]:

import csv
from pathlib import Path

results_file = Path("/content/experiment_results.csv")

with open(results_file, newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))

total = len(rows)
successful = sum(r["final_status"] == "PASS" for r in rows)
repairs = sum(int(r["repair_attempts"]) for r in rows)
initial_failures = sum(int(r["initial_tests_failed"]) for r in rows)
final_passes = sum(int(r["final_tests_passed"]) for r in rows)

print("=== WEEK 2 EVALUATION SUMMARY ===")
print("Experiments:", total)
print("Successful experiments:", successful)
print(f"Experiment success rate: {successful / total * 100:.1f}%" if total else "Experiment success rate: N/A")
print("Repair attempts:", repairs)
print("Initial test failures:", initial_failures)
print("Final passing test counts:", final_passes)

print("\n=== BUG TYPES TESTED ===")
for row in rows:
    print(f'{row["experiment_id"]}: {row["bug_type"]} -> {row["final_status"]}')


=== WEEK 2 EVALUATION SUMMARY ===
Experiments: 3
Successful experiments: 3
Experiment success rate: 100.0%
Repair attempts: 3
Initial test failures: 8
Final passing test counts: 10

=== BUG TYPES TESTED ===
EXP001: Off-by-one error and missing empty-input validation -> PASS
EXP002: Incorrect conditional logic -> PASS
EXP003: Undefined variable (NameError) -> PASS


In [39]:

import json
from pathlib import Path

bug_cases = [
    {
        "id": "TC01",
        "bug_type": "Off-by-one error",
        "function_name": "calculate_average",
        "buggy_code": """def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)
""",
        "tests": [
            "Normal average",
            "Single-element list",
            "Negative numbers",
            "Empty-list validation"
        ]
    },
    {
        "id": "TC02",
        "bug_type": "Incorrect conditional logic",
        "function_name": "is_adult",
        "buggy_code": """def is_adult(age):
    return age > 18
""",
        "tests": [
            "Adult age",
            "Exactly 18",
            "Under 18"
        ]
    },
    {
        "id": "TC03",
        "bug_type": "Undefined variable",
        "function_name": "calculate_total",
        "buggy_code": """def calculate_total(prices):
    total = sum(price for price in prices)
    return tota
""",
        "tests": [
            "Normal total",
            "Single price",
            "Empty prices"
        ]
    }
]

dataset_path = Path("/content/bug_cases.json")
dataset_path.write_text(
    json.dumps(bug_cases, indent=2),
    encoding="utf-8"
)

print("Dataset created:", dataset_path)
print("Bug cases:", len(bug_cases))
print(dataset_path.read_text())


Dataset created: /content/bug_cases.json
Bug cases: 3
[
  {
    "id": "TC01",
    "bug_type": "Off-by-one error",
    "function_name": "calculate_average",
    "buggy_code": "def calculate_average(numbers):\n    total = sum(numbers)\n    return total / (len(numbers) - 1)\n",
    "tests": [
      "Normal average",
      "Single-element list",
      "Negative numbers",
      "Empty-list validation"
    ]
  },
  {
    "id": "TC02",
    "bug_type": "Incorrect conditional logic",
    "function_name": "is_adult",
    "buggy_code": "def is_adult(age):\n    return age > 18\n",
    "tests": [
      "Adult age",
      "Exactly 18",
      "Under 18"
    ]
  },
  {
    "id": "TC03",
    "bug_type": "Undefined variable",
    "function_name": "calculate_total",
    "buggy_code": "def calculate_total(prices):\n    total = sum(price for price in prices)\n    return tota\n",
    "tests": [
      "Normal total",
      "Single price",
      "Empty prices"
    ]
  }
]


In [40]:

import json
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content")
DATASET_FILE = PROJECT_DIR / "bug_cases.json"
CASES_DIR = PROJECT_DIR / "evaluation_cases"

CASES_DIR.mkdir(exist_ok=True)

with open(DATASET_FILE, encoding="utf-8") as file:
    bug_cases = json.load(file)

# Fixed tests are written independently of Gemini's output.
test_definitions = {
    "TC01": '''import pytest
from solution import calculate_average

def test_normal_average():
    assert calculate_average([2, 4, 6, 8]) == 5.0

def test_single_element():
    assert calculate_average([10]) == 10

def test_negative_numbers():
    assert calculate_average([-2, -4, -6]) == -4

def test_empty_list():
    with pytest.raises(ValueError):
        calculate_average([])
''',
    "TC02": '''from solution import is_adult

def test_adult_age():
    assert is_adult(25) is True

def test_exactly_18():
    assert is_adult(18) is True

def test_under_18():
    assert is_adult(17) is False
''',
    "TC03": '''from solution import calculate_total

def test_normal_total():
    assert calculate_total([10, 20, 30]) == 60

def test_single_price():
    assert calculate_total([15]) == 15

def test_empty_prices():
    assert calculate_total([]) == 0
'''
}

for case in bug_cases:
    case_dir = CASES_DIR / case["id"]
    case_dir.mkdir(exist_ok=True)

    (case_dir / "solution.py").write_text(
        case["buggy_code"], encoding="utf-8"
    )
    (case_dir / "test_solution.py").write_text(
        test_definitions[case["id"]], encoding="utf-8"
    )

    # Run from this case's directory to avoid importing another solution.py.
    result = subprocess.run(
        [
            sys.executable, "-m", "pytest",
            "-q", "--tb=short", "test_solution.py"
        ],
        cwd=str(case_dir),
        capture_output=True,
        text=True,
        timeout=30
    )

    print(f'\n{case["id"]}: {case["bug_type"]}')
    print(result.stdout)
    print("Initial exit code:", result.returncode)

print("\nEvaluation case folders prepared.")



TC01: Off-by-one error
FFFF                                                                     [100%]
=================================== FAILURES ===================================
_____________________________ test_normal_average ______________________________
test_solution.py:5: in test_normal_average
    assert calculate_average([2, 4, 6, 8]) == 5.0
E   assert 6.666666666666667 == 5.0
E    +  where 6.666666666666667 = calculate_average([2, 4, 6, 8])
_____________________________ test_single_element ______________________________
test_solution.py:8: in test_single_element
    assert calculate_average([10]) == 10
           ^^^^^^^^^^^^^^^^^^^^^^^
solution.py:3: in calculate_average
    return total / (len(numbers) - 1)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
E   ZeroDivisionError: division by zero
____________________________ test_negative_numbers _____________________________
test_solution.py:11: in test_negative_numbers
    assert calculate_average([-2, -4, -6]) == -4
E   assert 

In [41]:

import csv
import json
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content")
CASES_DIR = PROJECT_DIR / "evaluation_cases"
DATASET_FILE = PROJECT_DIR / "bug_cases.json"
RUN_RESULTS = PROJECT_DIR / "evaluation_runner_results.csv"

with open(DATASET_FILE, encoding="utf-8") as f:
    cases = json.load(f)


def run_case_tests(case_dir):
    result = subprocess.run(
        [
            sys.executable, "-m", "pytest",
            "-q", "--tb=short", "test_solution.py"
        ],
        cwd=str(case_dir),
        capture_output=True,
        text=True,
        timeout=30
    )
    return result.returncode, result.stdout + "\n" + result.stderr


def ask_gemini_for_fix(code, feedback, function_name):
    prompt = f"""
You are a Python debugging agent.

Repair this code based on the test failures.

CURRENT CODE:
{code}

TEST FAILURE FEEDBACK:
{feedback}

Requirements:
- Preserve the function name: {function_name}
- Fix the demonstrated bug.
- Preserve behavior required by the tests.
- Do not change the tests.
- Return only complete Python code, preferably in a Python code block.
"""
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    if not response.text:
        raise ValueError("Gemini returned an empty response.")

    return extract_python_code(response.text)


results = []

for case in cases:
    case_id = case["id"]
    case_dir = CASES_DIR / case_id
    solution_file = case_dir / "solution.py"

    # Reset to the exact dataset baseline for a reproducible run.
    original_code = case["buggy_code"]
    solution_file.write_text(original_code, encoding="utf-8")

    attempts = 0
    final_status = "FAIL"
    final_output = ""

    print(f"\n{'=' * 55}")
    print(f'{case_id}: {case["bug_type"]}')

    try:
        for run_number in range(4):  # Initial test + at most 3 repairs
            exit_code, feedback = run_case_tests(case_dir)
            final_output = feedback

            print(f"\nTest run {run_number + 1}; exit code: {exit_code}")
            print(feedback[-900:])

            if exit_code == 0:
                final_status = "PASS"
                break

            if attempts >= 3:
                break

            attempts += 1
            print(f"Gemini repair attempt {attempts}/3")

            candidate = ask_gemini_for_fix(
                solution_file.read_text(encoding="utf-8"),
                feedback,
                case["function_name"]
            )

            # Syntax-check the candidate before saving it.
            compile(candidate, f"{case_id}_candidate.py", "exec")
            solution_file.write_text(candidate, encoding="utf-8")

    except Exception as error:
        final_output = str(error)
        print("Case error:", error)

    results.append({
        "experiment_id": case_id,
        "bug_type": case["bug_type"],
        "repair_attempts": attempts,
        "final_status": final_status,
        "details": final_output[-1000:].replace("\n", " ")
    })

    print("Final status:", final_status)

with open(RUN_RESULTS, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "experiment_id",
            "bug_type",
            "repair_attempts",
            "final_status",
            "details"
        ]
    )
    writer.writeheader()
    writer.writerows(results)

print("\n=== EVALUATION RUNNER SUMMARY ===")
print("Cases:", len(results))
print("Passed:", sum(r["final_status"] == "PASS" for r in results))
print("Failed:", sum(r["final_status"] == "FAIL" for r in results))
print("Results saved to:", RUN_RESULTS)



TC01: Off-by-one error

Test run 1; exit code: 1
o
____________________________ test_negative_numbers _____________________________
test_solution.py:11: in test_negative_numbers
    assert calculate_average([-2, -4, -6]) == -4
E   assert -6.0 == -4
E    +  where -6.0 = calculate_average([-2, -4, -6])
_______________________________ test_empty_list ________________________________
test_solution.py:14: in test_empty_list
    with pytest.raises(ValueError):
         ^^^^^^^^^^^^^^^^^^^^^^^^^
E   Failed: DID NOT RAISE <class 'ValueError'>
=========================== short test summary info ============================
FAILED test_solution.py::test_normal_average - assert 6.666666666666667 == 5.0
FAILED test_solution.py::test_single_element - ZeroDivisionError: division by...
FAILED test_solution.py::test_negative_numbers - assert -6.0 == -4
FAILED test_solution.py::test_empty_list - Failed: DID NOT RAISE <class 'Valu...
4 failed in 0.04s


Gemini repair attempt 1/3

Test run 2; exit code:

In [42]:
import pandas as pd

results = pd.read_csv("/content/evaluation_runner_results.csv")
display(results)

,experiment_id,bug_type,repair_attempts,final_status,details
0,TC01,Off-by-one error,1,PASS,.... ...
1,TC02,Incorrect conditional logic,1,PASS,... ...
2,TC03,Undefined variable,1,PASS,... ...


In [43]:
total = len(results)
passed = (results["final_status"] == "PASS").sum()
failed = (results["final_status"] == "FAIL").sum()

print("Total cases:", total)
print("Passed:", passed)
print("Failed:", failed)
print("Success rate:", round(passed / total * 100, 2), "%")
print("Total repair attempts:", results["repair_attempts"].sum())

Total cases: 3
Passed: 3
Failed: 0
Success rate: 100.0 %
Total repair attempts: 3


In [44]:
import json

def create_agent_result(
    case_id,
    bug_type,
    status,
    repair_attempts,
    message
):
    result = {
        "case_id": case_id,
        "bug_type": bug_type,
        "status": status,
        "repair_attempts": repair_attempts,
        "message": message
    }

    return json.dumps(result, indent=2)

print(create_agent_result(
    case_id="TC01",
    bug_type="Off-by-one error",
    status="PASS",
    repair_attempts=1,
    message="All tests passed after repair."
))

{
  "case_id": "TC01",
  "bug_type": "Off-by-one error",
  "status": "PASS",
  "repair_attempts": 1,
  "message": "All tests passed after repair."
}


In [45]:
sample_code = """
def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)
"""

sample_feedback = """
The average is incorrect.
A single-element list causes division by zero.
An empty list should raise ValueError.
"""

try:
    result = get_structured_repair(
        sample_code,
        sample_feedback,
        "calculate_average"
    )

    print(json.dumps(result, indent=2))

except Exception as e:
    print("Error:", type(e).__name__, str(e))

Error: NameError name 'get_structured_repair' is not defined


In [46]:
import json

def get_structured_repair(code, test_feedback, function_name):
    prompt = f"""
You are a Python debugging agent.

Analyze the buggy code and test failures.

Function name: {function_name}

Buggy code:
{code}

Test feedback:
{test_feedback}

Return a JSON object with exactly these fields:
- bug_type: string
- explanation: string
- fixed_code: string
- confidence: number between 0 and 1

Return valid JSON only. Do not use Markdown code fences.
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config={"response_mime_type": "application/json"}
    )

    result = json.loads(response.text)

    required_fields = {
        "bug_type",
        "explanation",
        "fixed_code",
        "confidence"
    }

    if not required_fields.issubset(result):
        raise ValueError("Missing required JSON fields.")

    if not isinstance(result["fixed_code"], str):
        raise ValueError("fixed_code must be a string.")

    if not isinstance(result["confidence"], (int, float)):
        raise ValueError("Confidence must be numeric.")

    if not 0 <= result["confidence"] <= 1:
        raise ValueError("Confidence must be between 0 and 1.")

    compile(result["fixed_code"], "<gemini_candidate>", "exec")

    return result

print("Structured repair function defined successfully!")

Structured repair function defined successfully!


In [47]:
sample_code = """
def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)
"""

sample_feedback = """
1. calculate_average([2, 4, 6, 8]) should return 5.0.
2. A single-element list should return its element.
3. Negative numbers should be averaged correctly.
4. An empty list should raise ValueError.
"""

try:
    result = get_structured_repair(
        sample_code,
        sample_feedback,
        "calculate_average"
    )

    print(json.dumps(result, indent=2))

except Exception as e:
    print("Error:", type(e).__name__, str(e))

{
  "bug_type": "Incorrect mathematical formula and edge case handling",
  "explanation": "The original code divides the sum by (len(numbers) - 1) instead of the total count of numbers, which is incorrect for calculating an arithmetic mean. Additionally, it does not handle empty lists (raising a ZeroDivisionError instead of a ValueError) or single-element lists correctly.",
  "fixed_code": "def calculate_average(numbers):\n    if not numbers:\n        raise ValueError(\"List is empty\")\n    return sum(numbers) / len(numbers)",
  "confidence": 1.0
}


In [48]:
from pathlib import Path
import subprocess
import sys

solution_path = Path("/content/evaluation_cases/TC01/solution.py")

# Save Gemini's generated code into the isolated TC01 case.
solution_path.write_text(result["fixed_code"], encoding="utf-8")

# Run the existing TC01 test suite.
test_run = subprocess.run(
    [
        sys.executable, "-m", "pytest",
        "-q", "--tb=short", "test_solution.py"
    ],
    cwd="/content/evaluation_cases/TC01",
    capture_output=True,
    text=True,
    timeout=30
)

print(test_run.stdout)
print(test_run.stderr)
print("Final test status:", "PASS" if test_run.returncode == 0 else "FAIL")

....                                                                     [100%]
4 passed in 0.01s


Final test status: PASS


In [49]:
print("run_debugging_agent exists:", "run_debugging_agent" in globals())
print("get_structured_repair exists:", "get_structured_repair" in globals())

run_debugging_agent exists: True
get_structured_repair exists: True


In [50]:
import inspect

print(inspect.getsource(run_debugging_agent))

def run_debugging_agent():
    """Run tests and attempt up to MAX_RETRIES repairs."""

    # Start from the current solution, not an old backup.
    for attempt in range(MAX_RETRIES + 1):
        code = SOLUTION_FILE.read_text()
        exit_code, feedback = run_tests()

        print(f"\n--- Test run {attempt + 1} ---")
        print(feedback[-1500:])

        if exit_code == 0:
            print("\nSUCCESS: All tests passed.")
            return True

        if attempt == MAX_RETRIES:
            break

        print(f"\nRepair attempt {attempt + 1} of {MAX_RETRIES}")

        try:
            candidate = request_repair(code, feedback)

            # Validate syntax before saving
            compile(candidate, "candidate_solution.py", "exec")

            # Save candidate and let pytest determine whether it works
            SOLUTION_FILE.write_text(candidate)
            print("Candidate saved. Running tests again...")

        except Exception as error:
            print("Repair a

In [51]:

def run_debugging_agent():
    """Run tests and repair code using Gemini's structured JSON output."""

    for attempt in range(MAX_RETRIES + 1):
        code = SOLUTION_FILE.read_text(encoding="utf-8")
        exit_code, feedback = run_tests()

        print(f"\n--- Test run {attempt + 1} ---")
        print(feedback[-1500:])

        if exit_code == 0:
            print("\nSUCCESS: All tests passed.")
            return True

        if attempt == MAX_RETRIES:
            break

        print(f"\nRepair attempt {attempt + 1} of {MAX_RETRIES}")

        try:
            # Extract the function name from the current code.
            import ast

            tree = ast.parse(code)
            functions = [
                node.name for node in tree.body
                if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            ]

            if not functions:
                raise ValueError("No function definition found in solution.")

            function_name = functions[0]

            # Ask Gemini for structured JSON repair.
            repair = get_structured_repair(
                code,
                feedback,
                function_name
            )

            print("Detected bug:", repair["bug_type"])
            print("Explanation:", repair["explanation"])
            print("Confidence:", repair["confidence"])

            candidate = repair["fixed_code"]

            # Validate syntax before saving.
            compile(candidate, "candidate_solution.py", "exec")

            # Save and verify with pytest on the next iteration.
            SOLUTION_FILE.write_text(candidate, encoding="utf-8")
            print("Candidate saved. Running tests again...")

        except Exception as error:
            print("Repair attempt failed:", error)

    print("\nSTOPPED: Retry limit reached or repair attempts failed.")
    return False


In [52]:
print("Solution file:", SOLUTION_FILE)
print("Test file:", TEST_FILE)
print("Maximum repairs:", MAX_RETRIES)
print("Solution exists:", SOLUTION_FILE.exists())
print("Test file exists:", TEST_FILE.exists())

Solution file: /content/solution.py
Test file: /content/test_average.py
Maximum repairs: 3
Solution exists: True
Test file exists: True


In [53]:
run_debugging_agent()


--- Test run 1 ---
...                                                                      [100%]
3 passed in 0.01s



SUCCESS: All tests passed.


True

In [54]:
from pathlib import Path

tc01_dir = Path("/content/evaluation_cases/TC01")
tc01_solution = tc01_dir / "solution.py"

# Save the known buggy version for this controlled test.
buggy_code = """
def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)
"""

tc01_solution.write_text(buggy_code.strip() + "\n", encoding="utf-8")

# Point the existing agent at TC01 and its tests.
SOLUTION_FILE = tc01_solution
TEST_FILE = tc01_dir / "test_solution.py"

print("Solution:", SOLUTION_FILE)
print("Tests:", TEST_FILE)
print("Both files exist:", SOLUTION_FILE.exists() and TEST_FILE.exists())

Solution: /content/evaluation_cases/TC01/solution.py
Tests: /content/evaluation_cases/TC01/test_solution.py
Both files exist: True


In [55]:
run_debugging_agent()


--- Test run 1 ---

>       assert calculate_average([10]) == 10
               ^^^^^^^^^^^^^^^^^^^^^^^

evaluation_cases/TC01/test_solution.py:8: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

numbers = [10]

    def calculate_average(numbers):
        total = sum(numbers)
>       return total / (len(numbers) - 1)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
E       ZeroDivisionError: division by zero

evaluation_cases/TC01/solution.py:3: ZeroDivisionError
____________________________ test_negative_numbers _____________________________

    def test_negative_numbers():
>       assert calculate_average([-2, -4, -6]) == -4
E       assert -6.0 == -4
E        +  where -6.0 = calculate_average([-2, -4, -6])

evaluation_cases/TC01/test_solution.py:11: AssertionError
_______________________________ test_empty_list ________________________________

    def test_empty_list():
>       with pytest.raises(ValueError):
             ^^^^^^^^^^^^^^^^^^^^^^^^^
E     

True

In [56]:
print("Maximum repair attempts:", MAX_RETRIES)
print("Agent function available:", callable(run_debugging_agent))
print("Structured repair available:", callable(get_structured_repair))

Maximum repair attempts: 3
Agent function available: True
Structured repair available: True


In [57]:
from unittest.mock import patch

with patch(
    "__main__.run_tests",
    return_value=(1, "Simulated test failure")
):
    with patch(
        "__main__.get_structured_repair",
        side_effect=ValueError("Simulated invalid repair response")
    ):
        result = run_debugging_agent()

print("\nExpected result: False")
print("Actual result:", result)
print("Retry limit:", MAX_RETRIES)


--- Test run 1 ---
Simulated test failure

Repair attempt 1 of 3
Repair attempt failed: Simulated invalid repair response

--- Test run 2 ---
Simulated test failure

Repair attempt 2 of 3
Repair attempt failed: Simulated invalid repair response

--- Test run 3 ---
Simulated test failure

Repair attempt 3 of 3
Repair attempt failed: Simulated invalid repair response

--- Test run 4 ---
Simulated test failure

STOPPED: Retry limit reached or repair attempts failed.

Expected result: False
Actual result: False
Retry limit: 3


In [58]:
print("Real pytest runner:", run_tests.__module__)
print("Solution file:", SOLUTION_FILE)
print("Test file:", TEST_FILE)
print("Maximum repairs:", MAX_RETRIES)

Real pytest runner: __main__
Solution file: /content/evaluation_cases/TC01/solution.py
Test file: /content/evaluation_cases/TC01/test_solution.py
Maximum repairs: 3


In [59]:

from pathlib import Path

cli_code = r'''
import argparse
import json
import subprocess
import sys
from pathlib import Path

BASE_DIR = Path("/content")
CASES_DIR = BASE_DIR / "evaluation_cases"


def main():
    parser = argparse.ArgumentParser(
        description="Agentic AI Software Debugging and Test Automation"
    )
    parser.add_argument(
        "--case",
        required=True,
        help="Test case ID, for example TC01, TC02, or TC03"
    )
    args = parser.parse_args()

    case_id = args.case.upper()
    case_dir = CASES_DIR / case_id
    solution = case_dir / "solution.py"
    tests = case_dir / "test_solution.py"

    if not solution.is_file() or not tests.is_file():
        print(json.dumps({
            "case_id": case_id,
            "status": "ERROR",
            "message": "Case or required files not found."
        }, indent=2))
        return 2

    # Run the existing notebook agent through a generated runner.
    runner = BASE_DIR / "_agent_cli_runner.py"
    runner.write_text(
        """
import sys
from pathlib import Path

sys.path.insert(0, str(Path('/content')))
""",
        encoding="utf-8"
    )

    print(f"Selected case: {case_id}")
    print(f"Solution: {solution}")
    print(f"Tests: {tests}")
    print()
    print("The CLI entry point is ready.")
    print("The agent integration will be connected in the next step.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''

Path("/content/agent_cli.py").write_text(cli_code, encoding="utf-8")
print("CLI file created:", Path("/content/agent_cli.py").exists())


CLI file created: True


In [60]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "/content/agent_cli.py", "--case", "TC01"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

Selected case: TC01
Solution: /content/evaluation_cases/TC01/solution.py
Tests: /content/evaluation_cases/TC01/test_solution.py

The CLI entry point is ready.
The agent integration will be connected in the next step.


Exit code: 0


In [61]:
from pathlib import Path

agent_module = r'''
import ast
import json
import os
import subprocess
import sys
from pathlib import Path

from google import genai

BASE_DIR = Path("/content")
CASES_DIR = BASE_DIR / "evaluation_cases"
MAX_RETRIES = 3
MODEL = "gemini-3.5-flash-lite"


def get_client():
    key = os.environ.get("GEMINI_API_KEY")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            key = None

    if not key:
        raise RuntimeError("GEMINI_API_KEY is not configured.")

    return genai.Client(api_key=key)


def run_tests(case_dir):
    result = subprocess.run(
        [
            sys.executable, "-m", "pytest",
            "-q", "--tb=short", "test_solution.py"
        ],
        cwd=str(case_dir),
        capture_output=True,
        text=True,
        timeout=30
    )
    return result.returncode, result.stdout + result.stderr


def request_structured_repair(client, code, feedback, function_name):
    prompt = f"""
You are a Python debugging agent.

Function name: {function_name}

Current code:
{code}

Pytest feedback:
{feedback}

Return JSON with exactly these fields:
bug_type (string), explanation (string),
fixed_code (string), confidence (number from 0 to 1).

Return only valid JSON. Preserve the function name.
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"response_mime_type": "application/json"}
    )

    data = json.loads(response.text)
    required = {"bug_type", "explanation", "fixed_code", "confidence"}

    if not isinstance(data, dict) or not required.issubset(data):
        raise ValueError("Invalid structured repair response.")

    if not isinstance(data["fixed_code"], str):
        raise ValueError("fixed_code must be a string.")

    if not isinstance(data["confidence"], (int, float)):
        raise ValueError("confidence must be numeric.")

    if not 0 <= data["confidence"] <= 1:
        raise ValueError("confidence must be between 0 and 1.")

    tree = ast.parse(data["fixed_code"])
    function_names = {
        node.name for node in tree.body
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
    }

    if function_name not in function_names:
        raise ValueError("Repair did not preserve the required function.")

    return data


def main():
    import argparse

    parser = argparse.ArgumentParser(
        description="Agentic AI Debugging and Test Automation"
    )
    parser.add_argument("--case", required=True, choices=["TC01", "TC02", "TC03"])
    args = parser.parse_args()

    case_dir = CASES_DIR / args.case
    solution = case_dir / "solution.py"
    test_file = case_dir / "test_solution.py"

    if not solution.is_file() or not test_file.is_file():
        print(json.dumps({
            "case_id": args.case,
            "status": "ERROR",
            "message": "Solution or test file is missing."
        }, indent=2))
        return 2

    try:
        client = get_client()
        attempts = 0

        for run_number in range(MAX_RETRIES + 1):
            code = solution.read_text(encoding="utf-8")
            exit_code, feedback = run_tests(case_dir)

            print(f"Test run {run_number + 1}: exit code {exit_code}")
            print(feedback[-1000:])

            if exit_code == 0:
                print(json.dumps({
                    "case_id": args.case,
                    "status": "PASS",
                    "repair_attempts": attempts,
                    "message": "All tests passed."
                }, indent=2))
                return 0

            if attempts >= MAX_RETRIES:
                break

            tree = ast.parse(code)
            functions = [
                node.name for node in tree.body
                if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            ]
            if not functions:
                raise ValueError("No function found in solution.")

            attempts += 1
            print(f"Repair attempt {attempts}/{MAX_RETRIES}")

            repair = request_structured_repair(
                client, code, feedback, functions[0]
            )

            print("Bug type:", repair["bug_type"])
            compile(repair["fixed_code"], "<candidate>", "exec")
            solution.write_text(repair["fixed_code"], encoding="utf-8")

        print(json.dumps({
            "case_id": args.case,
            "status": "FAIL",
            "repair_attempts": attempts,
            "message": "Retry limit reached."
        }, indent=2))
        return 1

    except Exception as error:
        print(json.dumps({
            "case_id": args.case,
            "status": "ERROR",
            "message": str(error)
        }, indent=2))
        return 2


if __name__ == "__main__":
    raise SystemExit(main())
'''

Path("/content/debugging_agent.py").write_text(
    agent_module, encoding="utf-8"
)
print("Standalone agent module created.")

Standalone agent module created.


In [62]:
from pathlib import Path

Path("/content/agent_cli.py").write_text(
    '''import subprocess
import sys

if __name__ == "__main__":
    result = subprocess.run([
        sys.executable,
        "/content/debugging_agent.py",
        *sys.argv[1:]
    ])
    raise SystemExit(result.returncode)
''',
    encoding="utf-8"
)

print("CLI connected to debugging agent.")

CLI connected to debugging agent.


In [63]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "/content/agent_cli.py",
        "--case", "TC01"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print("Errors / warnings:")
    print(result.stderr)

print("CLI exit code:", result.returncode)

{
  "case_id": "TC01",
  "status": "ERROR",
  "message": "GEMINI_API_KEY is not configured."
}

CLI exit code: 2


In [64]:
from pathlib import Path

agent_path = Path("/content/debugging_agent.py")
source = agent_path.read_text(encoding="utf-8")

old_function = '''def get_client():
    key = os.environ.get("GEMINI_API_KEY")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            key = None

    if not key:
        raise RuntimeError("GEMINI_API_KEY is not configured.")

    return genai.Client(api_key=key)
'''

new_function = '''def get_client():
    key = os.environ.get("GEMINI_API_KEY")

    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            key = None

    if not key:
        raise RuntimeError(
            "GEMINI_API_KEY is unavailable. Check Colab Secrets "
            "or configure the environment variable before launching."
        )

    return genai.Client(api_key=key)
'''

if old_function not in source:
    print("The expected function was not found. No file changes made.")
else:
    agent_path.write_text(
        source.replace(old_function, new_function, 1),
        encoding="utf-8"
    )
    print("API key loading function updated.")

API key loading function updated.


In [65]:
import os
import subprocess
import sys
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise RuntimeError(
        "Add GEMINI_API_KEY to Google Colab Secrets first."
    )

env = os.environ.copy()
env["GEMINI_API_KEY"] = api_key

result = subprocess.run(
    [
        sys.executable,
        "/content/agent_cli.py",
        "--case",
        "TC01"
    ],
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("CLI exit code:", result.returncode)

Test run 1: exit code 0
....                                                                     [100%]
4 passed in 0.01s

{
  "case_id": "TC01",
  "status": "PASS",
  "repair_attempts": 0,
  "message": "All tests passed."
}

CLI exit code: 0


In [66]:
from pathlib import Path
import os
import subprocess
import sys
from google.colab import userdata

case_dir = Path("/content/evaluation_cases/TC01")
solution = case_dir / "solution.py"

buggy_code = """
def calculate_average(numbers):
    total = sum(numbers)
    return total / (len(numbers) - 1)
"""

# Reset only the isolated TC01 example.
solution.write_text(buggy_code.strip() + "\n", encoding="utf-8")

api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("GEMINI_API_KEY is unavailable in Colab Secrets.")

env = os.environ.copy()
env["GEMINI_API_KEY"] = api_key

result = subprocess.run(
    [
        sys.executable,
        "/content/agent_cli.py",
        "--case", "TC01"
    ],
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout)
if result.stderr:
    print("Errors / warnings:\n", result.stderr)

print("CLI exit code:", result.returncode)

Test run 1: exit code 1
tal / (len(numbers) - 1)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
E   ZeroDivisionError: division by zero
____________________________ test_negative_numbers _____________________________
test_solution.py:11: in test_negative_numbers
    assert calculate_average([-2, -4, -6]) == -4
E   assert -6.0 == -4
E    +  where -6.0 = calculate_average([-2, -4, -6])
_______________________________ test_empty_list ________________________________
test_solution.py:14: in test_empty_list
    with pytest.raises(ValueError):
         ^^^^^^^^^^^^^^^^^^^^^^^^^
E   Failed: DID NOT RAISE <class 'ValueError'>
=========================== short test summary info ============================
FAILED test_solution.py::test_normal_average - assert 6.666666666666667 == 5.0
FAILED test_solution.py::test_single_element - ZeroDivisionError: division by...
FAILED test_solution.py::test_negative_numbers - assert -6.0 == -4
FAILED test_solution.py::test_empty_list - Failed: DID NOT RAISE <class 'Va

In [67]:
import os
import subprocess
import sys
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
env = os.environ.copy()
env["GEMINI_API_KEY"] = api_key

result = subprocess.run(
    [
        sys.executable,
        "/content/agent_cli.py",
        "--case", "TC02"
    ],
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("CLI exit code:", result.returncode)

Test run 1: exit code 0
...                                                                      [100%]
3 passed in 0.01s

{
  "case_id": "TC02",
  "status": "PASS",
  "repair_attempts": 0,
  "message": "All tests passed."
}

CLI exit code: 0


In [68]:
import os
import subprocess
import sys
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
env = os.environ.copy()
env["GEMINI_API_KEY"] = api_key

result = subprocess.run(
    [
        sys.executable,
        "/content/agent_cli.py",
        "--case", "TC03"
    ],
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("CLI exit code:", result.returncode)

Test run 1: exit code 0
...                                                                      [100%]
3 passed in 0.01s

{
  "case_id": "TC03",
  "status": "PASS",
  "repair_attempts": 0,
  "message": "All tests passed."
}

CLI exit code: 0


In [69]:
from pathlib import Path

tc02_solution = Path(
    "/content/evaluation_cases/TC02/solution.py"
)

buggy_code = """
def is_adult(age):
    return age > 18
"""

tc02_solution.write_text(
    buggy_code.strip() + "\n",
    encoding="utf-8"
)

print("TC02 reset to its original buggy version.")
print(tc02_solution.read_text())

TC02 reset to its original buggy version.
def is_adult(age):
    return age > 18



In [70]:
import os
import subprocess
import sys
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("GEMINI_API_KEY is missing from Colab Secrets.")

env = os.environ.copy()
env["GEMINI_API_KEY"] = api_key

result = subprocess.run(
    [
        sys.executable,
        "/content/agent_cli.py",
        "--case", "TC02"
    ],
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("CLI exit code:", result.returncode)

Test run 1: exit code 1
.F.                                                                      [100%]
=================================== FAILURES ===================================
_______________________________ test_exactly_18 ________________________________
test_solution.py:7: in test_exactly_18
    assert is_adult(18) is True
E   assert False is True
E    +  where False = is_adult(18)
=========================== short test summary info ============================
FAILED test_solution.py::test_exactly_18 - assert False is True
1 failed, 2 passed in 0.03s

Repair attempt 1/3
Bug type: Logical Error
Test run 2: exit code 0
...                                                                      [100%]
3 passed in 0.01s

{
  "case_id": "TC02",
  "status": "PASS",
  "repair_attempts": 1,
  "message": "All tests passed."
}

CLI exit code: 0


In [71]:
from pathlib import Path

tc03_solution = Path(
    "/content/evaluation_cases/TC03/solution.py"
)

buggy_code = """
def calculate_total(prices):
    total = sum(prices)
    return tota
"""

tc03_solution.write_text(
    buggy_code.strip() + "\n",
    encoding="utf-8"
)

print("TC03 reset to its original buggy version.")
print(tc03_solution.read_text())

TC03 reset to its original buggy version.
def calculate_total(prices):
    total = sum(prices)
    return tota



In [72]:
import os
import subprocess
import sys
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("GEMINI_API_KEY is missing from Colab Secrets.")

env = os.environ.copy()
env["GEMINI_API_KEY"] = api_key

result = subprocess.run(
    [
        sys.executable,
        "/content/agent_cli.py",
        "--case", "TC03"
    ],
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

print("CLI exit code:", result.returncode)

Test run 1: exit code 1
           ^^^^
E   NameError: name 'tota' is not defined
______________________________ test_single_price _______________________________
test_solution.py:7: in test_single_price
    assert calculate_total([15]) == 15
           ^^^^^^^^^^^^^^^^^^^^^
solution.py:3: in calculate_total
    return tota
           ^^^^
E   NameError: name 'tota' is not defined
______________________________ test_empty_prices _______________________________
test_solution.py:10: in test_empty_prices
    assert calculate_total([]) == 0
           ^^^^^^^^^^^^^^^^^^^
solution.py:3: in calculate_total
    return tota
           ^^^^
E   NameError: name 'tota' is not defined
=========================== short test summary info ============================
FAILED test_solution.py::test_normal_total - NameError: name 'tota' is not de...
FAILED test_solution.py::test_single_price - NameError: name 'tota' is not de...
FAILED test_solution.py::test_empty_prices - NameError: name 'tota' is not 

In [73]:
import csv
from pathlib import Path

results = [
    {
        "case_id": "TC01",
        "bug_type": "Off-by-one error",
        "initial_failures": 4,
        "tests_passed": 4,
        "repair_attempts": 1,
        "status": "PASS"
    },
    {
        "case_id": "TC02",
        "bug_type": "Incorrect conditional logic",
        "initial_failures": 1,
        "tests_passed": 3,
        "repair_attempts": 1,
        "status": "PASS"
    },
    {
        "case_id": "TC03",
        "bug_type": "Undefined variable",
        "initial_failures": 3,
        "tests_passed": 3,
        "repair_attempts": 1,
        "status": "PASS"
    }
]

output_file = Path("/content/week2_evaluation_results.csv")

with output_file.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print("Saved:", output_file)
print("Cases:", len(results))
print("Passed:", sum(r["status"] == "PASS" for r in results))
print("Total repair attempts:", sum(r["repair_attempts"] for r in results))

Saved: /content/week2_evaluation_results.csv
Cases: 3
Passed: 3
Total repair attempts: 3


In [74]:
from pathlib import Path

files_to_check = [
    "/content/debugging_agent.py",
    "/content/agent_cli.py",
    "/content/bug_cases.json",
    "/content/week2_evaluation_results.csv",
    "/content/evaluation_cases/TC01/test_solution.py",
    "/content/evaluation_cases/TC02/test_solution.py",
    "/content/evaluation_cases/TC03/test_solution.py",
]

for filename in files_to_check:
    path = Path(filename)
    print(
        "FOUND" if path.is_file() else "MISSING",
        filename
    )

FOUND /content/debugging_agent.py
FOUND /content/agent_cli.py
FOUND /content/bug_cases.json
FOUND /content/week2_evaluation_results.csv
FOUND /content/evaluation_cases/TC01/test_solution.py
FOUND /content/evaluation_cases/TC02/test_solution.py
FOUND /content/evaluation_cases/TC03/test_solution.py


In [75]:

from pathlib import Path

readme = r"""
# Agentic AI Software Debugging and Test Automation System

## 1. Project overview
This project demonstrates an AI debugging agent that uses Gemini to
propose Python code repairs and pytest to verify whether the repairs work.

## 2. Week 2 objectives
- Connect to the Gemini API.
- Run automated tests using pytest.
- Generate structured JSON repair responses.
- Validate generated Python syntax before saving a repair.
- Retry repairs up to a configurable limit.
- Provide a command-line interface (CLI).
- Record evaluation results in CSV format.

## 3. Architecture
1. The user selects a test case through the CLI.
2. The agent runs pytest on the selected case.
3. If tests fail, the agent sends the code and failure feedback to Gemini.
4. Gemini returns JSON containing the bug type, explanation,
   fixed code, and confidence score.
5. The agent validates the response and Python syntax.
6. The candidate is saved and pytest runs again.
7. The agent stops when tests pass or the repair limit is reached.

## 4. Project files
- `debugging_agent.py`: standalone debugging agent.
- `agent_cli.py`: command-line entry point.
- `bug_cases.json`: bug-case dataset.
- `evaluation_cases/`: isolated solutions and tests for TC01–TC03.
- `week2_evaluation_results.csv`: observed Week 2 evaluation summary.

## 5. Setup
Install the dependencies in your Python environment:

```bash
pip install google-genai pytest
```

Configure `GEMINI_API_KEY` as an environment variable before running
the standalone CLI. Do not commit real API keys to GitHub.

## 6. Run the CLI
```bash
python agent_cli.py --case TC01
python agent_cli.py --case TC02
python agent_cli.py --case TC03
```

The current implementation uses the configured case files under
`/content/evaluation_cases/`. Update `CASES_DIR` in `debugging_agent.py`
if running outside the current Colab directory structure.

## 7. Week 2 evaluation
Three controlled cases were tested:
- TC01: off-by-one error — PASS after one repair attempt.
- TC02: incorrect conditional logic — PASS after one repair attempt.
- TC03: undefined variable — PASS after one repair attempt.

Total: 3 cases passed; 3 repair attempts; 10 tests passed after repair.

These are preliminary controlled results, not a broad benchmark.

## 8. Limitations and safety
- Gemini-generated code is not guaranteed to be correct.
- Syntax validation alone does not establish correctness or safety.
- The current prototype runs generated code through pytest; do not use
  it with untrusted code without appropriate isolation.
- Test results depend on the selected tests and may miss other defects.
- Week 2 does not yet implement RAG, persistent memory, a graphical UI,
  or a complete production sandbox.

## 9. Next steps
Expand evaluation coverage, strengthen execution isolation, add logging,
and complete the remaining project lifecycle requirements.
"""

Path("/content/README.md").write_text(readme.strip() + "\n", encoding="utf-8")
print("README created:", Path("/content/README.md").is_file())
print("Location: /content/README.md")


README created: True
Location: /content/README.md


In [76]:
from pathlib import Path
import shutil

source = Path("/content")
project = source / "agentic-ai-debugging-test-automation"

project.mkdir(exist_ok=True)

# Main project files
files = [
    "debugging_agent.py",
    "agent_cli.py",
    "bug_cases.json",
    "README.md",
    "week2_evaluation_results.csv",
]

for name in files:
    src = source / name
    if src.exists():
        shutil.copy2(src, project / name)
        print("Copied:", name)
    else:
        print("MISSING:", name)

# Copy test cases and their isolated source files
cases_src = source / "evaluation_cases"
if cases_src.exists():
    shutil.copytree(
        cases_src,
        project / "evaluation_cases",
        dirs_exist_ok=True
    )
    print("Copied evaluation cases")

print("\nProject folder:", project)

Copied: debugging_agent.py
Copied: agent_cli.py
Copied: bug_cases.json
Copied: README.md
Copied: week2_evaluation_results.csv
Copied evaluation cases

Project folder: /content/agentic-ai-debugging-test-automation


In [77]:
from pathlib import Path

project = Path("/content/agentic-ai-debugging-test-automation")

gitignore = """
.env
.env.*
!.env.example
__pycache__/
*.py[cod]
.venv/
venv/
.ipynb_checkpoints/
"""

(project / ".gitignore").write_text(
    gitignore.strip() + "\n",
    encoding="utf-8"
)

print(".gitignore created")

.gitignore created


In [78]:
from pathlib import Path
import shutil

source = Path("/content")
project = source / "agentic-ai-debugging-test-automation"

project.mkdir(exist_ok=True)

files = [
    "debugging_agent.py",
    "agent_cli.py",
    "bug_cases.json",
    "README.md",
    "week2_evaluation_results.csv"
]

for name in files:
    file_path = source / name
    if file_path.exists():
        shutil.copy2(file_path, project / name)
        print("Copied:", name)
    else:
        print("Not found:", name)

cases = source / "evaluation_cases"

if cases.exists():
    shutil.copytree(cases, project / "evaluation_cases", dirs_exist_ok=True)
    print("Copied evaluation_cases folder")

print("\nYour project folder is:", project)

Copied: debugging_agent.py
Copied: agent_cli.py
Copied: bug_cases.json
Copied: README.md
Copied: week2_evaluation_results.csv
Copied evaluation_cases folder

Your project folder is: /content/agentic-ai-debugging-test-automation


In [79]:
import shutil

project = "/content/agentic-ai-debugging-test-automation"
zip_path = "/content/agentic-ai-debugging-test-automation.zip"

shutil.make_archive(
    "/content/agentic-ai-debugging-test-automation",
    "zip",
    root_dir=project
)

print("ZIP file created:", zip_path)

ZIP file created: /content/agentic-ai-debugging-test-automation.zip


In [80]:
from google.colab import files

files.download("/content/agentic-ai-debugging-test-automation.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>